In [ ]:
train='/content/flight_delays_train.csv'
test='/content/flight_delays_test.csv'

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df=pd.read_csv(train)
df.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732,N
1,c-4,c-20,c-3,1548,US,PIT,MCO,834,N
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416,N
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872,N
4,c-10,c-7,c-6,1828,WN,MDW,OMA,423,Y


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Month                 100000 non-null  object 
 1   DayofMonth            100000 non-null  object 
 2   DayOfWeek             100000 non-null  object 
 3   DepTime               100000 non-null  int64  
 4   UniqueCarrier         100000 non-null  object 
 5   Origin                100000 non-null  object 
 6   Dest                  100000 non-null  object 
 7   Distance              100000 non-null  int64  
 8   dep_delayed_15min     100000 non-null  int8   
 9   HourOfDay             100000 non-null  int64  
 10  Route                 100000 non-null  object 
 11  LogDistance           100000 non-null  float64
 12  CarrierDelayRate      100000 non-null  float64
 13  RouteDelayRate        100000 non-null  float64
 14  HourlyRouteDelayRate  100000 non-null  float64
 15  T

In [ ]:
df.duplicated().any()

np.False_

In [ ]:
df.isna().any()

,0
Month,False
DayofMonth,False
DayOfWeek,False
DepTime,False
UniqueCarrier,False
Origin,False
Dest,False
Distance,False
dep_delayed_15min,False


In [ ]:
y = (df['dep_delayed_15min'] == 'Y').astype(int)
X = df.drop(columns=['dep_delayed_15min'])

### Preprocessing

In [ ]:
# Convert "c-*" columns to numeric
def cat_to_num(df):
  for col in ['Month','DayofMonth','DayOfWeek']:
      df[col] = df[col].str.replace('c-','').astype(int)
  return df
cat_to_num(X)

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance
0,8,21,7,1934,AA,ATL,DFW,732
1,4,20,3,1548,US,PIT,MCO,834
2,9,2,5,1422,XE,RDU,CLE,416
3,11,25,6,1015,OO,DEN,MEM,872
4,10,7,6,1828,WN,MDW,OMA,423
...,...,...,...,...,...,...,...,...
99995,5,4,3,1618,OO,SFO,RDD,199
99996,1,18,3,804,CO,EWR,DAB,884
99997,1,24,2,1901,NW,DTW,IAH,1076
99998,4,27,4,1515,MQ,DFW,GGG,140


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [ ]:
# num_cols = ['Month','DayofMonth','DayOfWeek','DepTime','Distance']
# cat_cols = ['UniqueCarrier','Origin','Dest']

# preprocessor = ColumnTransformer([
#     ('num', 'passthrough', num_cols),
#     ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
# ])
num_cols = ['Month','DayofMonth','DayOfWeek','DepTime','Distance']
cat_cols = ['UniqueCarrier','Origin','Dest']

preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

### Model Training

In [ ]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score

# model = XGBClassifier(
#     n_estimators=300,
#     max_depth=6,
#     learning_rate=0.05,
#     subsample=0.9,
#     colsample_bytree=0.9,
#     eval_metric='logloss',
#     use_label_encoder=False
# )

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=3,
    gamma=0.5,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    predictor="gpu_predictor",
    eval_metric='logloss'
)

In [ ]:
pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', model)
])

In [ ]:
pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [07:38:46] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['Month', 'DayofMonth',
                                                   'DayOfWeek', 'DepTime',
                                                   'Distance']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['UniqueCarrier', 'Origin',
                                                   'Dest'])])),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.9...
                               feature_types=None, feature_weights=None,
                               gamma=0.5, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None, min_child_weight=3,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=400,
                               n_jobs=None, num_parallel_tree=None, ...))])

In [ ]:
# probas = pipeline.predict_proba(X_test)
# y_proba = probas[:, 1]

from sklearn.calibration import CalibratedClassifierCV
calibrated = CalibratedClassifierCV(pipeline, method='sigmoid', cv=3)
calibrated.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [07:38:49] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [07:38:51] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [07:38:54] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


CalibratedClassifierCV(cv=3,
                       estimator=Pipeline(steps=[('prep',
                                                  ColumnTransformer(transformers=[('num',
                                                                                   'passthrough',
                                                                                   ['Month',
                                                                                    'DayofMonth',
                                                                                    'DayOfWeek',
                                                                                    'DepTime',
                                                                                    'Distance']),
                                                                                  ('cat',
                                                                                   OneHotEncoder(handle_unknown='ignore'),
                                                                                   ['UniqueCarrier',
                                                                                    'Origin',
                                                                                    'Dest'])])),
                                                 ('model',
                                                  XGBClassifier(base_score=None,
                                                                booster=None,
                                                                callbacks=None,
                                                                colsample_bylevel=None,
                                                                cols...
                                                                feature_weights=None,
                                                                gamma=0.5,
                                                                grow_policy=None,
                                                                importance_type=None,
                                                                interaction_constraints=None,
                                                                learning_rate=0.05,
                                                                max_bin=None,
                                                                max_cat_threshold=None,
                                                                max_cat_to_onehot=None,
                                                                max_delta_step=None,
                                                                max_depth=6,
                                                                max_leaves=None,
                                                                min_child_weight=3,
                                                                missing=nan,
                                                                monotone_constraints=None,
                                                                multi_strategy=None,
                                                                n_estimators=400,
                                                                n_jobs=None,
                                                                num_parallel_tree=None, ...))]))

In [ ]:
# print('ROC AUC:', roc_auc_score(y_test, y_proba))

proba = calibrated.predict_proba(X_test)[:,1]
print("Final Accuracy:", roc_auc_score(y_test, proba))
pred=calibrated.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

Final Accuracy: 0.7495251089891268
Accuracy: 0.8232


## Test Dataset

In [ ]:
df_test=pd.read_csv(test)
df_test.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance
0,c-7,c-25,c-3,615,YV,MRY,PHX,598
1,c-4,c-17,c-2,739,WN,LAS,HOU,1235
2,c-12,c-2,c-7,651,MQ,GSP,ORD,577
3,c-3,c-25,c-7,1614,WN,BWI,MHT,377
4,c-6,c-6,c-3,1505,UA,ORD,STL,258


In [ ]:
cat_to_num(df_test)

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance
0,7,25,3,615,YV,MRY,PHX,598
1,4,17,2,739,WN,LAS,HOU,1235
2,12,2,7,651,MQ,GSP,ORD,577
3,3,25,7,1614,WN,BWI,MHT,377
4,6,6,3,1505,UA,ORD,STL,258
...,...,...,...,...,...,...,...,...
99995,6,5,2,852,WN,CRP,HOU,187
99996,11,24,6,1446,UA,ORD,LAS,1515
99997,1,30,2,1509,OO,ORD,SGF,438
99998,1,5,5,804,DL,LGA,ATL,761


In [ ]:
dep_delayed_15min=calibrated.predict_proba(df_test)[:,1]
# dep_delayed_15min=pipeline.predict_proba(df_test)[:,1]

In [ ]:
submission = pd.DataFrame({
    'id': range(len(df_test)),
    'dep_delayed_15min': dep_delayed_15min
})

In [ ]:
submission.to_csv('submission.csv', index=False)